In [0]:
%sql CREATE OR REPLACE VIEW gold.kpi_mrr AS
SELECT
  MES_ANO,
  SUM(VALOR_PAGO_MES) AS MRR
FROM gold.fato_resumo_usuario_mensal
GROUP BY MES_ANO
ORDER BY MES_ANO


In [0]:
%sql
CREATE OR REPLACE VIEW gold.metrica_novos_usuarios AS
SELECT
  date_format(DATA_CADASTRO, 'yyyy-MM') AS MES_CADASTRO,
  COUNT(ID_USUARIO) AS NOVOS_USUARIOS
FROM gold.dim_usuario
GROUP BY date_format(DATA_CADASTRO, 'yyyy-MM')
ORDER BY MES_CADASTRO


In [0]:
%sql
CREATE OR REPLACE VIEW gold.metrica_minutos_assistidos AS
SELECT
  MES_ANO,
  SUM(MINUTOS_ASSISTIDOS_MES) AS TOTAL_MINUTOS_ASSISTIDOS
FROM gold.fato_resumo_usuario_mensal
GROUP BY MES_ANO
ORDER BY MES_ANO


In [0]:
%sql
CREATE OR REPLACE VIEW gold.kpi_churn_rate AS
WITH atividade_usuario_mes AS (
  SELECT DISTINCT FK_USUARIO, MES_ANO, FLAG_ATIVO_MES
  FROM gold.fato_resumo_usuario_mensal
),
atividade_com_mes_anterior AS (
  SELECT
    FK_USUARIO,
    MES_ANO,
    FLAG_ATIVO_MES,
    LAG(FLAG_ATIVO_MES, 1, 0) OVER (PARTITION BY FK_USUARIO ORDER BY MES_ANO) AS ATIVO_MES_ANTERIOR
  FROM atividade_usuario_mes
),
churn_events AS (
  SELECT
    MES_ANO,
    SUM(ATIVO_MES_ANTERIOR) AS TOTAL_ATIVOS_MES_ANTERIOR,
    SUM(CASE WHEN ATIVO_MES_ANTERIOR = 1 AND FLAG_ATIVO_MES = 0 THEN 1 ELSE 0 END) AS CHURNED_USERS
  FROM atividade_com_mes_anterior
  GROUP BY MES_ANO
)
SELECT
  MES_ANO,
  (CHURNED_USERS / TOTAL_ATIVOS_MES_ANTERIOR) * 100 AS TAXA_DE_CHURN_PERCENTUAL
FROM churn_events
WHERE TOTAL_ATIVOS_MES_ANTERIOR > 0
ORDER BY MES_ANO


In [0]:
%sql
CREATE OR REPLACE VIEW gold.kpi_ltv AS
WITH arpu AS (
  SELECT
    k.MES_ANO,
    k.MRR / a.TOTAL_ATIVOS_MES_ANTERIOR AS ARPU_MENSAL
  FROM gold.kpi_mrr k
  JOIN (
    SELECT MES_ANO, SUM(ATIVO_MES_ANTERIOR) AS TOTAL_ATIVOS_MES_ANTERIOR
    FROM (
      SELECT
        MES_ANO,
        LAG(FLAG_ATIVO_MES, 1, 0) OVER (PARTITION BY FK_USUARIO ORDER BY MES_ANO) AS ATIVO_MES_ANTERIOR
      FROM gold.fato_resumo_usuario_mensal
    )
    GROUP BY MES_ANO
  ) a ON k.MES_ANO = a.MES_ANO
  WHERE a.TOTAL_ATIVOS_MES_ANTERIOR > 0
),
churn AS (
  SELECT
    MES_ANO,
    TAXA_DE_CHURN_PERCENTUAL / 100 AS CHURN_RATE
  FROM gold.kpi_churn_rate
)
SELECT
  a.MES_ANO,
  a.ARPU_MENSAL / c.CHURN_RATE AS LTV_ESTIMADO
FROM arpu a
JOIN churn c ON a.MES_ANO = c.MES_ANO
WHERE c.CHURN_RATE > 0
ORDER BY a.MES_ANO


In [0]:
%sql
CREATE OR REPLACE VIEW gold.kpi_engajamento_pais_plano AS
SELECT
  f.MES_ANO,
  u.PAIS,
  p.NOME_PLANO,
  AVG(f.MINUTOS_ASSISTIDOS_MES) AS MEDIA_MINUTOS_ASSISTIDOS
FROM gold.fato_resumo_usuario_mensal f
JOIN gold.dim_usuario u ON f.FK_USUARIO = u.ID_USUARIO
JOIN gold.dim_plano p ON f.FK_PLANO = p.ID_PLANO
WHERE f.FLAG_ATIVO_MES = 1
GROUP BY f.MES_ANO, u.PAIS, p.NOME_PLANO
ORDER BY f.MES_ANO, u.PAIS, MEDIA_MINUTOS_ASSISTIDOS DESC


In [0]:
%sql 
SELECT * FROM gold.kpi_mrr;


MES_ANO,MRR
2019-01,11149.579999999996
2019-02,9440.859999999988
2019-03,10976.020000000015
2019-04,11425.919999999996
2019-05,11247.820000000002
2019-06,11490.21
2019-07,10623.340000000004
2019-08,11244.149999999996
2019-09,11119.35
2019-10,10843.7


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT * FROM gold.kpi_churn_rate;


MES_ANO,TAXA_DE_CHURN_PERCENTUAL
2019-02,34.78260869565217
2019-03,40.35087719298245
2019-04,26.923076923076923
2019-05,28.125
2019-06,36.69724770642202
2019-07,27.450980392156865
2019-08,30.303030303030305
2019-09,30.15075376884422
2019-10,33.83838383838384
2019-11,28.57142857142857


In [0]:
%sql
SELECT * FROM gold.kpi_ltv;


MES_ANO,LTV_ESTIMADO
2019-02,1130.9363541666653
2019-03,461.04137067059753
2019-04,523.939894179894
2019-05,408.08417233560095
2019-06,279.5609129464285
2019-07,248.07250000000008
2019-08,222.1897904191616
2019-09,184.39588750000001
2019-10,158.64139204965275
2019-11,169.85862068965517


In [0]:
%sql
SELECT * FROM gold.metrica_novos_usuarios;


MES_CADASTRO,NOVOS_USUARIOS
2018-01,237
2018-02,216
2018-03,256
2018-04,254
2018-05,265
2018-06,237
2018-07,217
2018-08,248
2018-09,220
2018-10,248


In [0]:
%sql
SELECT * FROM gold.metrica_minutos_assistidos;


MES_ANO,TOTAL_MINUTOS_ASSISTIDOS
2019-01,50836.0
2019-02,45826.0
2019-03,56222.0
2019-04,47542.0
2019-05,50959.0
2019-06,55412.0
2019-07,55705.0
2019-08,51108.0
2019-09,47779.0
2019-10,46440.0


In [0]:
%sql
SELECT * FROM gold.kpi_engajamento_pais_plano;


MES_ANO,PAIS,NOME_PLANO,MEDIA_MINUTOS_ASSISTIDOS
2019-01,Afghanistan,Ultra,0.0
2019-01,Afghanistan,Padrão,0.0
2019-01,Algeria,Básico,0.0
2019-01,Andorra,Padrão,0.0
2019-01,Anguilla,Premium,0.0
2019-01,Antigua and Barbuda,Básico,0.0
2019-01,Argentina,Ultra,0.0
2019-01,Armenia,Ultra,0.0
2019-01,Armenia,Básico,0.0
2019-01,Aruba,Premium,0.0
